# CrowdFundAI: Exploratory Data Analysis & Feature Engineering

This notebook performs EDA and Feature Engineering on **331,675 authentic Kickstarter crowdfunding projects**.
All engineered features are strictly **pre-launch features** to prevent target leakage.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
data_path = os.path.join("..", "data", "processed", "kickstarter_cleaned.csv")
df = pd.read_csv(data_path)
print(f"Loaded dataset with {len(df):,} rows and {len(df.columns)} columns.")

## Step 1: Dataset Summary & Data Types

In [ ]:
df.info()

## Step 2: Missing Value Analysis

In [ ]:
print("Missing values count per column:")
print(df.isnull().sum())

## Step 3: Target Variable Distribution (`is_successful`)

In [ ]:
counts = df["is_successful"].value_counts()
props = df["is_successful"].value_counts(normalize=True)
print(f"Failed (0): {counts.get(0, 0):,} ({props.get(0, 0):.2%})")
print(f"Successful (1): {counts.get(1, 0):,} ({props.get(1, 0):.2%})")

plt.figure(figsize=(6, 4))
sns.barplot(x=["Failed (0)", "Successful (1)"], y=[counts.get(0, 0), counts.get(1, 0)], hue=["Failed (0)", "Successful (1)"], palette=["#e74c3c", "#2ecc71"], legend=False)
plt.title("Crowdfunding Outcome Distribution")
plt.ylabel("Campaign Count")
plt.show()

## Step 4: Numerical Features Analysis

In [ ]:
df[["usd_goal_real", "duration_days", "name_length", "name_word_count"]].describe().T

## Step 5: Categorical Analysis (Category Success Rates)

In [ ]:
cat_df = df.groupby("main_category")["is_successful"].agg(["count", "mean"]).sort_values(by="mean", ascending=False)
print(cat_df)

plt.figure(figsize=(10, 5))
cat_sorted = df.groupby("main_category")["is_successful"].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=cat_sorted, x="main_category", y="is_successful", hue="main_category", palette="viridis", legend=False)
plt.axhline(df["is_successful"].mean(), color="red", linestyle="--", label="Average Success Rate")
plt.title("Success Rate by Main Category")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.show()

## Step 6: Correlation Analysis

In [ ]:
corr_cols = ["is_successful", "usd_goal_real", "log_usd_goal", "duration_days", "name_length", "name_word_count"]
plt.figure(figsize=(7, 5))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.show()

## Step 7: Feature Engineering

In [ ]:
df_feat = df.copy()
df_feat["goal_per_day"] = df_feat["usd_goal_real"] / (df_feat["duration_days"] + 1.0)
df_feat["log_goal_per_day"] = np.log1p(df_feat["goal_per_day"])
df_feat["category_success_rate"] = df_feat.groupby("main_category")["is_successful"].transform("mean")
cat_med_goal = df_feat.groupby("main_category")["usd_goal_real"].transform("median")
df_feat["goal_to_cat_median_ratio"] = df_feat["usd_goal_real"] / (cat_med_goal + 1.0)
df_feat["log_goal_to_cat_median_ratio"] = np.log1p(df_feat["goal_to_cat_median_ratio"])
df_feat["is_usd_currency"] = (df_feat["currency"] == "USD").astype(int)
df_feat["is_top_country"] = (df_feat["country"].isin(["US", "GB", "CA"])).astype(int)
df_feat["is_weekend_launch"] = (df_feat["launched_day_of_week"] >= 5).astype(int)
df_feat["launched_quarter"] = ((df_feat["launched_month"] - 1) // 3 + 1).astype(int)
print(f"Engineered features added. Shape: {df_feat.shape}")

## Step 8: Final Feature Selection & Export

In [ ]:
out_path = os.path.join("..", "data", "processed", "model_features.csv")
df_feat.to_csv(out_path, index=False)
print(f"Saved training feature set to {out_path} with shape {df_feat.shape}.")